In [19]:
import pandas as pd
import numpy as np
from pathlib import Path
import cv2
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 200)
pd.set_option("display.width", 200)


# =========================
# 1. 読み込み
# =========================

vod_title = "M8 vs. EDG - VALORANT Masters Santiago - SWISS"
video_path = Path(f"../../data/vods/{vod_title}.mp4")
csv_path = Path(f"../../data/processed/round_labels/{vod_title}_round_labels_fixed.csv")
df = pd.read_csv(csv_path)

print(df.shape)
display(df.head())
display(df.tail())

(44, 32)


,value,start_sec,end_sec,n_samples,mean_diff,min_diff,max_diff,duration_sec,t_sec,t_min,left_score,right_score,left_score_candidates,right_score_candidates,score_total,round_no_from_score,map_no,round_no,left_win_label,round_ocr_start_samples,round_ocr_end_samples,round_no_ocr_start,round_no_ocr_end,prev_round_no_ocr,prev_end_like,gap_from_prev,ocr_step,sequence_status,action_candidate,fix_applied,needs_review,map
0,True,9.009,78.078,70.0,22.858459,17.960205,26.488715,69.069,9.009,0.150150,0.0,0.0,"[0, 0, 0]","[0, 0, 0]",0.0,1.0,1,1,1.0,"[{'t_sec': 10.009, 'offset': 1.0, 'round_no': ...","[{'t_sec': 73.078, 'offset': -5.0, 'round_no':...",1,1,NaN,NaN,NaN,NaN,first,keep,NaN,False,HAVEN
1,True,106.106,203.203,98.0,23.621082,19.780490,27.049723,97.097,106.106,1.768433,1.0,0.0,"[1, 1, 1]","[0, 0, 0]",1.0,2.0,1,2,0.0,"[{'t_sec': 107.106, 'offset': 1.0, 'round_no':...","[{'t_sec': 198.203, 'offset': -5.0, 'round_no'...",2,2,1.0,78.078,28.028,1.0,ok,keep,NaN,False,HAVEN
2,True,235.235,278.278,44.0,25.698283,21.291341,27.700168,43.043,235.235,3.920583,1.0,1.0,"[1, 1, 1]","[1, 1, 1]",2.0,3.0,1,3,0.0,"[{'t_sec': 236.235, 'offset': 1.0, 'round_no':...","[{'t_sec': 273.278, 'offset': -5.0, 'round_no'...",3,3,2.0,203.203,32.032,1.0,ok,keep,NaN,False,HAVEN
3,True,297.297,390.390,94.0,25.161743,19.347548,27.986844,93.093,297.297,4.954950,1.0,2.0,"[1, 1, 1]","[2, 2, 2]",3.0,4.0,1,4,1.0,"[{'t_sec': 298.297, 'offset': 1.0, 'round_no':...","[{'t_sec': 385.39, 'offset': -5.0, 'round_no':...",4,4,3.0,278.278,19.019,1.0,ok,keep,NaN,False,HAVEN
4,True,421.421,515.515,95.0,23.683746,19.510579,27.125407,94.094,421.421,7.023683,2.0,2.0,"[2, 2, 2]","[2, 2, 2]",4.0,5.0,1,5,0.0,"[{'t_sec': 422.421, 'offset': 1.0, 'round_no':...","[{'t_sec': 510.515, 'offset': -5.0, 'round_no'...",5,5,4.0,390.390,31.031,1.0,ok,keep,NaN,False,HAVEN


,value,start_sec,end_sec,n_samples,mean_diff,min_diff,max_diff,duration_sec,t_sec,t_min,left_score,right_score,left_score_candidates,right_score_candidates,score_total,round_no_from_score,map_no,round_no,left_win_label,round_ocr_start_samples,round_ocr_end_samples,round_no_ocr_start,round_no_ocr_end,prev_round_no_ocr,prev_end_like,gap_from_prev,ocr_step,sequence_status,action_candidate,fix_applied,needs_review,map
39,True,5150.145,5252.247,103.0,18.012072,13.363254,21.365506,102.102,5150.145,85.835750,7.0,8.0,"[7, 7, 7]","[8, 8, 8]",15.0,16.0,2,16,0.0,"[{'t_sec': 5151.145, 'offset': 1.0, 'round_no'...","[{'t_sec': 5247.247, 'offset': -5.0, 'round_no...",16,16,15.0,5135.130,15.015,1.0,ok,keep,NaN,False,PEARL
40,True,5344.339,5435.430,92.0,18.075620,9.574761,24.011013,91.091,5344.339,89.072317,7.0,9.0,"[7, 7, 7]","[9, 9, 9]",16.0,17.0,2,17,0.0,"[{'t_sec': 5345.339, 'offset': 1.0, 'round_no'...","[{'t_sec': 5430.43, 'offset': -5.0, 'round_no'...",17,17,16.0,5252.247,92.092,1.0,ok,keep,NaN,False,PEARL
41,True,5469.464,5574.569,106.0,14.659329,10.677382,17.938368,105.105,5469.464,91.157733,7.0,10.0,"[7, 7, 7]","[10, 10, 10]",17.0,18.0,2,18,0.0,"[{'t_sec': 5470.464, 'offset': 1.0, 'round_no'...","[{'t_sec': 5569.569, 'offset': -5.0, 'round_no...",18,18,17.0,5435.430,34.034,1.0,ok,keep,NaN,False,PEARL
42,True,5679.674,5764.759,86.0,14.349154,9.363173,17.445123,85.085,5679.674,94.661233,7.0,11.0,"[7, 7, 7]","[11, 11, 11]",18.0,19.0,2,19,0.0,"[{'t_sec': 5680.674, 'offset': 1.0, 'round_no'...","[{'t_sec': 5759.759, 'offset': -5.0, 'round_no...",19,19,19.0,5607.602,72.072,0.0,drop_previous_candidate,drop_previous,drop_shorter_previous,False,PEARL
43,True,5796.791,5897.892,102.0,13.513559,9.481961,17.052979,101.101,5796.791,96.613183,7.0,12.0,"[7, 7, 7]","[12, 12, 12]",19.0,20.0,2,20,NaN,"[{'t_sec': 5797.791, 'offset': 1.0, 'round_no'...","[{'t_sec': 5892.892, 'offset': -5.0, 'round_no...",20,20,19.0,5764.759,32.032,1.0,ok,keep,NaN,False,PEARL


In [20]:
print(df.columns.tolist())

['value', 'start_sec', 'end_sec', 'n_samples', 'mean_diff', 'min_diff', 'max_diff', 'duration_sec', 't_sec', 't_min', 'left_score', 'right_score', 'left_score_candidates', 'right_score_candidates', 'score_total', 'round_no_from_score', 'map_no', 'round_no', 'left_win_label', 'round_ocr_start_samples', 'round_ocr_end_samples', 'round_no_ocr_start', 'round_no_ocr_end', 'prev_round_no_ocr', 'prev_end_like', 'gap_from_prev', 'ocr_step', 'sequence_status', 'action_candidate', 'fix_applied', 'needs_review', 'map']


In [22]:
df[df["needs_review"]==True]

,value,start_sec,end_sec,n_samples,mean_diff,min_diff,max_diff,duration_sec,t_sec,t_min,left_score,right_score,left_score_candidates,right_score_candidates,score_total,round_no_from_score,map_no,round_no,left_win_label,round_ocr_start_samples,round_ocr_end_samples,round_no_ocr_start,round_no_ocr_end,prev_round_no_ocr,prev_end_like,gap_from_prev,ocr_step,sequence_status,action_candidate,fix_applied,needs_review,map
24,True,3242.239,3284.281,43.0,22.978553,18.324354,24.937337,42.042,3242.239,54.037317,0.0,0.0,"[0, 0, 0]","[0, 0, 0]",0.0,1.0,2,1,1.0,"[{'t_sec': 3243.239, 'offset': 1.0, 'round_no'...","[{'t_sec': 3279.281, 'offset': -5.0, 'round_no...",1,1,24.0,3229.226,13.013,-23.0,map_change_or_ocr_error,manual_check,manual_review_required,True,PEARL
33,NaN,4269.265,4269.265,NaN,NaN,NaN,NaN,0.000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2,10,NaN,NaN,NaN,10,10,9.0,NaN,0.000,5.0,inserted_missing_round,inserted,insert_missing_round_between,True,NaN


In [23]:
df

,value,start_sec,end_sec,n_samples,mean_diff,min_diff,max_diff,duration_sec,t_sec,t_min,left_score,right_score,left_score_candidates,right_score_candidates,score_total,round_no_from_score,map_no,round_no,left_win_label,round_ocr_start_samples,round_ocr_end_samples,round_no_ocr_start,round_no_ocr_end,prev_round_no_ocr,prev_end_like,gap_from_prev,ocr_step,sequence_status,action_candidate,fix_applied,needs_review,map
0,True,9.009,78.078,70.0,22.858459,17.960205,26.488715,69.069,9.009,0.150150,0.0,0.0,"[0, 0, 0]","[0, 0, 0]",0.0,1.0,1,1,1.0,"[{'t_sec': 10.009, 'offset': 1.0, 'round_no': ...","[{'t_sec': 73.078, 'offset': -5.0, 'round_no':...",1,1,NaN,NaN,NaN,NaN,first,keep,NaN,False,HAVEN
1,True,106.106,203.203,98.0,23.621082,19.780490,27.049723,97.097,106.106,1.768433,1.0,0.0,"[1, 1, 1]","[0, 0, 0]",1.0,2.0,1,2,0.0,"[{'t_sec': 107.106, 'offset': 1.0, 'round_no':...","[{'t_sec': 198.203, 'offset': -5.0, 'round_no'...",2,2,1.0,78.078,28.028,1.0,ok,keep,NaN,False,HAVEN
2,True,235.235,278.278,44.0,25.698283,21.291341,27.700168,43.043,235.235,3.920583,1.0,1.0,"[1, 1, 1]","[1, 1, 1]",2.0,3.0,1,3,0.0,"[{'t_sec': 236.235, 'offset': 1.0, 'round_no':...","[{'t_sec': 273.278, 'offset': -5.0, 'round_no'...",3,3,2.0,203.203,32.032,1.0,ok,keep,NaN,False,HAVEN
3,True,297.297,390.390,94.0,25.161743,19.347548,27.986844,93.093,297.297,4.954950,1.0,2.0,"[1, 1, 1]","[2, 2, 2]",3.0,4.0,1,4,1.0,"[{'t_sec': 298.297, 'offset': 1.0, 'round_no':...","[{'t_sec': 385.39, 'offset': -5.0, 'round_no':...",4,4,3.0,278.278,19.019,1.0,ok,keep,NaN,False,HAVEN
4,True,421.421,515.515,95.0,23.683746,19.510579,27.125407,94.094,421.421,7.023683,2.0,2.0,"[2, 2, 2]","[2, 2, 2]",4.0,5.0,1,5,0.0,"[{'t_sec': 422.421, 'offset': 1.0, 'round_no':...","[{'t_sec': 510.515, 'offset': -5.0, 'round_no'...",5,5,4.0,390.390,31.031,1.0,ok,keep,NaN,False,HAVEN
5,True,542.542,650.650,109.0,23.825269,19.197130,27.194607,108.108,542.542,9.042367,2.0,3.0,"[2, 2, 2]","[3, 3, 3]",5.0,6.0,1,6,0.0,"[{'t_sec': 543.542, 'offset': 1.0, 'round_no':...","[{'t_sec': 645.65, 'offset': -5.0, 'round_no':...",6,6,5.0,515.515,27.027,1.0,ok,keep,NaN,False,HAVEN
6,True,736.736,862.862,127.0,24.199966,18.810683,27.099338,126.126,736.736,12.278933,2.0,4.0,"[2, 2, 2]","[4, 4, 4]",6.0,7.0,1,7,0.0,"[{'t_sec': 737.736, 'offset': 1.0, 'round_no':...","[{'t_sec': 857.862, 'offset': -5.0, 'round_no'...",7,7,6.0,650.650,86.086,1.0,ok,keep,NaN,False,HAVEN
7,True,895.895,972.972,78.0,25.151095,18.837484,26.630968,77.077,895.895,14.931583,2.0,5.0,"[2, 2, 2]","[5, 5, 5]",7.0,8.0,1,8,0.0,"[{'t_sec': 896.895, 'offset': 1.0, 'round_no':...","[{'t_sec': 967.972, 'offset': -5.0, 'round_no'...",8,8,7.0,862.862,33.033,1.0,ok,keep,NaN,False,HAVEN
8,True,991.991,1148.147,157.0,23.723775,18.561252,27.512126,156.156,991.991,16.533183,2.0,6.0,"[2, 2, 2]","[6, 6, 6]",8.0,9.0,1,9,0.0,"[{'t_sec': 992.991, 'offset': 1.0, 'round_no':...","[{'t_sec': 1143.147, 'offset': -5.0, 'round_no...",9,9,8.0,972.972,19.019,1.0,ok,keep,NaN,False,HAVEN
9,True,1170.169,1262.261,93.0,21.988577,18.152751,26.461480,92.092,1170.169,19.502817,2.0,7.0,"[2, 2, 2]","[7, 7, 7]",9.0,10.0,1,10,1.0,"[{'t_sec': 1171.169, 'offset': 1.0, 'round_no'...","[{'t_sec': 1257.261, 'offset': -5.0, 'round_no...",10,10,9.0,1148.147,22.022,1.0,ok,keep,NaN,False,HAVEN
